In [1]:
from datasets import load_dataset 
from random import randrange
 
 
# 从hub加载数据集并得到一个样本
dataset = load_dataset("databricks/databricks-dolly-15k", split="train", cache_dir="/root/autodl-fs/data")
sample = dataset[randrange(len(dataset))]

Using the latest cached version of the dataset since databricks/databricks-dolly-15k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/autodl-fs/data/databricks___databricks-dolly-15k/default/0.0.0/bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a (last modified on Thu Jun 13 22:42:25 2024).


In [3]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

model_name_or_path = "/root/autodl-tmp/llama_code/merged_model_directory"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name_or_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name_or_path, trust_remote_code=True)
model = model.eval()
# input_text = '类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领'
# response, history = model.chat(tokenizer=tokenizer, query=input_text, history=[])
# print(response)

prompt = f"""### Instruction:
Use the Input below to create an instruction, which could have been used to generate the input using an LLM. 
 
### Input:
{sample['response']}
 
### Response:
"""
 
# input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids.cuda()
input_ids = tokenizer(prompt, return_tensors="pt", truncation=True)


outputs = model.generate(input_ids=input_ids, max_new_tokens=100, do_sample=True, top_p=0.9,temperature=0.9)

print(f"Prompt:\n{sample['response']}\n")
print(f"Generated instruction:\n{tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0][len(prompt):]}")
print(f"Ground truth:\n{sample['instruction']}")

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


AttributeError: 

In [4]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer


# model_dir = "models/llama-7-int4-dolly"
model_dir = "/root/autodl-tmp/llama_code/models/demo/llama-7-int4-dolly"
 
# 加载基础LLM模型与分词器
model = AutoPeftModelForCausalLM.from_pretrained(
    model_dir,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    load_in_4bit=True,
) 
tokenizer = AutoTokenizer.from_pretrained(model_dir)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OSError: /root/autodl-tmp/llama_code/models/demo/llama-7-int4-dolly does not appear to have a file named config.json. Checkout 'https://huggingface.co//root/autodl-tmp/llama_code/models/demo/llama-7-int4-dolly/None' for available files.

In [ ]:
prompt = f"""### Instruction:
Use the Input below to create an instruction, which could have been used to generate the input using an LLM. 
 
### Input:
{sample['response']}
 
### Response:
"""
 
input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids.cuda()

outputs = model.generate(input_ids=input_ids, max_new_tokens=100, do_sample=True, top_p=0.9,temperature=0.9)

print(f"Prompt:\n{sample['response']}\n")
print(f"Generated instruction:\n{tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0][len(prompt):]}")
print(f"Ground truth:\n{sample['instruction']}")

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name_or_path = "/root/autodl-tmp/llama_code/merged_model_directory"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name_or_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name_or_path, 
                                  trust_remote_code=True,
                                  low_cpu_mem_usage=True,
                                  torch_dtype=torch.float16,
                                  load_in_4bit=True)
model = model.eval()


prompt = f"""### Instruction:
Use the Input below to create an instruction, which could have been used to generate the input using an LLM. 
 
### Input:
{sample['response']}
 
### Response:
"""
 
input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids.cuda()

outputs = model.generate(input_ids=input_ids, max_new_tokens=100, do_sample=True, top_p=0.9,temperature=0.9)

print(f"Prompt:\n{sample['response']}\n")
print(f"Generated instruction:\n{tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0][len(prompt):]}")
print(f"Ground truth:\n{sample['instruction']}")

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/root/miniconda3/envs/glm/lib/python3.10/site-packages/bitsandbytes/nn/modules.py:426: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Prompt:
Global warming is caused by the release of greenhouses gases into the atmosphere. Greenhouse gases in the atmosphere trap the heat from the sun and lead to an overall increase in temperature on the planet. Greenhouse gases are mostly produced by burning fossil fuels like coal, oil, and natural gas. Deforestation is also a contributing factor to global warming since trees capture carbon from the atmosphere when growing, so cutting them down or burning them both releases carbon they’ve been storing, as well as removes their ability to clean the air.

Generated instruction:
What is global warming?

Ground truth:
What causes global warming?
